In [1]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("/content/abstracts.csv")
texts = df["text"].tolist()
labels = df["label"].values

In [3]:
encoder = SentenceTransformer("all-MiniLM-L6-v2")
X_embed = encoder.encode(texts, show_progress_bar=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

In [4]:
pca = PCA(n_components=4)
X_pca = pca.fit_transform(X_embed)

scaler = MinMaxScaler(feature_range=(0, np.pi))
X_scaled = scaler.fit_transform(X_pca)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, labels, test_size=0.25, random_state=42
)

In [7]:
pip install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 120.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 137.4 MB/s eta 0:00:00


In [8]:
import pennylane as qml
import torch

In [9]:
n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

In [10]:
@qml.qnode(dev, interface="torch")
def quantum_circuit(x, weights):
    # Encoding
    for i in range(n_qubits):
        qml.RY(x[i], wires=i)

    # Entanglement
    for i in range(n_qubits - 1):
        qml.CNOT(wires=[i, i + 1])

    # Variational layer
    for i in range(n_qubits):
        qml.RY(weights[i, 0], wires=i)
        qml.RZ(weights[i, 1], wires=i)

    return qml.expval(qml.PauliZ(0))

In [11]:
weights = torch.randn((n_qubits, 2), requires_grad=True)
optimizer = torch.optim.SGD([weights], lr=0.01)

In [12]:
def sigmoid(x):
    return 1 / (1 + torch.exp(-x))

In [13]:
def loss_fn(y_pred, y_true):
    return torch.mean((y_pred - y_true) ** 2)

In [14]:
epochs = 20

for epoch in range(epochs):
    total_loss = 0
    for x, y in zip(X_train, y_train):
        x_t = torch.tensor(x, dtype=torch.float32)
        y_t = torch.tensor(y, dtype=torch.float32)

        optimizer.zero_grad()
        out = quantum_circuit(x_t, weights)
        pred = sigmoid(out)
        loss = loss_fn(pred, y_t)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(X_train):.4f}")

Epoch 1/20 | Loss: 0.3856
Epoch 2/20 | Loss: 0.3709
Epoch 3/20 | Loss: 0.3530
Epoch 4/20 | Loss: 0.3327
Epoch 5/20 | Loss: 0.3111
Epoch 6/20 | Loss: 0.2897
Epoch 7/20 | Loss: 0.2696
Epoch 8/20 | Loss: 0.2514
Epoch 9/20 | Loss: 0.2353
Epoch 10/20 | Loss: 0.2212
Epoch 11/20 | Loss: 0.2089
Epoch 12/20 | Loss: 0.1982
Epoch 13/20 | Loss: 0.1889
Epoch 14/20 | Loss: 0.1807
Epoch 15/20 | Loss: 0.1736
Epoch 16/20 | Loss: 0.1675
Epoch 17/20 | Loss: 0.1623
Epoch 18/20 | Loss: 0.1578
Epoch 19/20 | Loss: 0.1540
Epoch 20/20 | Loss: 0.1509


In [15]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [16]:
y_preds = []

for x in X_test:
    x_t = torch.tensor(x, dtype=torch.float32)
    out = quantum_circuit(x_t, weights)
    y_preds.append(int(sigmoid(out).item() > 0.5))

print("Accuracy:", accuracy_score(y_test, y_preds))
print("Precision:", precision_score(y_test, y_preds))
print("Recall:", recall_score(y_test, y_preds))
print("F1:", f1_score(y_test, y_preds))

Accuracy: 0.8536585365853658
Precision: 0.76
Recall: 1.0
F1: 0.8636363636363636
